In [1]:
import numpy, scipy, matplotlib, pennylane, netket, jax
print(netket.__version__, pennylane.__version__, jax.devices())

∣NK⟩ Tip: Debug multi-node HPC? `djaxrun -np 2 python Examples/Sharding/multi_process.py`

3.23.0 0.45.1 [CpuDevice(id=0)]


In [1]:
import numpy as np
import netket as nk

∣NK⟩ Tip: NetKet 3.22 has plenty of new stuff! Check it out at https://www.netket.org/posts/07-netket322/

In [4]:
# Pauli matrices

sx = np.array([[0,1],[1,0]], dtype=complex)
sy = np.array([[0,-1j],[1j,0]], dtype=complex)
sz = np.array([[1,0],[0,-1]], dtype=complex)

# Spin operator: hbar = 1.0

Sx = sx / 2
Sy = sy / 2
Sz = sz / 2

# Spin-up state along z
up = np.array([1,0], dtype=complex)

# 1.  Check the commutation relation [Sx,Sy] = iSz
commutator = Sx @ Sy - Sy @ Sx
print("Commutator [Sx, Sy]:" , commutator)
print("iSz:", 1j * Sz)

# 2. Expectation value <O> = <psi|O|psi> 

def expectation(state, O):
    return np.vdot(state, O @ state)

print("<Sx>:", expectation(up, Sx))
print("<Sy>:", expectation(up, Sy))
print("<Sz>:", expectation(up, Sz))

# 3. Uncertainty : Delta S = sqrt(<S^2> - <S>^2)

def uncertainty(state,S):
    mean = expectation(state, S)
    mean_square = np.vdot(state, S @ S @ state).real
    return np.sqrt(max(0, mean_square - mean**2))

print("Delta Sx:", uncertainty(up, Sx))
print("Delta Sy:", uncertainty(up, Sy))
print("Delta Sz:", uncertainty(up, Sz))

print("Product =", uncertainty(up, Sx) * uncertainty(up, Sy))

print("lower bound =", 0.5 * abs(expectation(up, Sz)))

Commutator [Sx, Sy]: [[0.+0.5j 0.+0.j ]
 [0.+0.j  0.-0.5j]]
iSz: [[ 0.+0.5j  0.+0.j ]
 [ 0.+0.j  -0.-0.5j]]
<Sx>: 0j
<Sy>: 0j
<Sz>: (0.5+0j)
Delta Sx: (0.5+0j)
Delta Sy: (0.5+0j)
Delta Sz: 0.0
Product = (0.25+0j)
lower bound = 0.25


In [7]:
# 1. Define a Hilbert space containing ONE spin-1/2
hi = nk.hilbert.Spin(s=1/2,N=1)

# 2. Construct the three Pauli operators  
sx = nk.operator.spin.sigmax(hi,0)
sy = nk.operator.spin.sigmay(hi,0)
sz = nk.operator.spin.sigmaz(hi,0) # it applies to the first spin (index 0)

# 3. Convert the operators into matrices
X = sx.to_dense()
Y = sy.to_dense()
Z = sz.to_dense()

print("Pauli X:\n", X)
print("Pauli Y:\n", Y)
print("Pauli Z:\n", Z)

# 4. Check the commutator [X,Y] = 2iZ
print("Commutator [X,Y]:\n", X @ Y - Y @ X)
print("2iZ:\n", 2j * Z)

# 5. Find eigenvalues of the pauli operators

print("Eigenvalues of X:", np.linalg.eigvals(X))
print("Eigenvalues of Y:", np.linalg.eigvals(Y))
print("Eigenvalues of Z:", np.linalg.eigvals(Z))

# 6. Define spin operators S = sigma/2
Sx = sx / 2
Sy = sy / 2
Sz = sz / 2


Pauli X:
 [[0. 1.]
 [1. 0.]]
Pauli Y:
 [[0.+0.j 0.-1.j]
 [0.+1.j 0.+0.j]]
Pauli Z:
 [[ 1.  0.]
 [ 0. -1.]]
Commutator [X,Y]:
 [[0.+2.j 0.+0.j]
 [0.+0.j 0.-2.j]]
2iZ:
 [[ 0.+2.j  0.+0.j]
 [ 0.+0.j -0.-2.j]]
Eigenvalues of X: [ 1.+0.j -1.+0.j]
Eigenvalues of Y: [ 1.+0.j -1.+0.j]
Eigenvalues of Z: [ 1.+0.j -1.+0.j]


# Practice Problem

# Practical Challenge: Simulate a Two-Qubit Quantum Sensor

## Scenario

You are building a tiny quantum sensor with two spin-1/2 particles. The sensor starts in a known quantum state. The particles interact with each other, and an external magnetic field affects their spins. Your task is to predict the measurement statistics after the system evolves for a given time.

Use **NetKet + NumPy** only. No VQE or neural networks yet.

## Your Tasks

1. Create a two-spin Hilbert space using NetKet.
2. Construct the Pauli operators for each spin.
3. Build the Hamiltonian:

   $$H = J\sigma_1^z\sigma_2^z + h(\sigma_1^x+\sigma_2^x)$$

4. Initialize the system in the state:

   $$\vert{}\psi(0)\rangle = \vert{}\uparrow\uparrow\rangle$$

5. Calculate the evolved state at time $t$:

   $$\vert{}\psi(t)\rangle = e^{-iHt}\vert{}\psi(0)\rangle$$

6. Calculate the probability of measuring each of the four possible spin configurations.
7. Calculate the expectation value of the first spin's $z$-component:

   $$\langle S_z^{(1)}\rangle = \left\langle\psi(t)\left\vert{}\frac{\sigma_z^{(1)}}{2}\right\vert{}\psi(t)\right\rangle$$

8. Verify that the total measurement probability equals 1.

## Sample Input

Use these parameter values in your program:

```python
J = 1.0
h = 0.5
t = 1.0
initial_state = "|up up>"


## Expected Output Format

Your program should calculate the numerical results.

```text
Two-spin quantum sensor
-----------------------
Hilbert-space dimension: 4

Initial state: |up up>
Evolution time: 1.0

Measurement probabilities:
|up up>       : [your result]
|up down>     : [your result]
|down up>     : [your result]
|down down>   : [your result]

Expectation value <Sz(1)>: [your result]

Total probability: approximately 1.000000
Normalization check: PASSED

In [5]:

import netket as nk
import numpy as np
from scipy.linalg import expm

# 1. Create a Hilbert space for two spin-1/2 particles
hi = nk.hilbert.Spin(s=1/2, N=2)

# 2. Function to construct Pauli operator matrices
def operator_matrix(op, spin_index):
    if op == 'x':
        return nk.operator.spin.sigmax(hi, spin_index).to_dense()
    elif op == 'y':
        return nk.operator.spin.sigmay(hi, spin_index).to_dense()
    elif op == 'z':
        return nk.operator.spin.sigmaz(hi, spin_index).to_dense()
    else:
        raise ValueError("op must be 'x', 'y', or 'z'")

# 3. Define the parameters
J = 1.0
h = 0.5
t = 1.0

# 4. Construct the Pauli operators for each spin
X0 = operator_matrix('x', 0)
X1 = operator_matrix('x', 1)
Z0 = operator_matrix('z', 0)
Z1 = operator_matrix('z', 1)

# 5. Construct the Hamiltonian
H = J * (Z0 @ Z1) + h * (X0 + X1)

# 6. Define the initial state |up up>
up = np.array([1, 0], dtype=complex)
up_up = np.kron(up, up)

# 7. Calculate the evolved state |psi(t)> = exp(-iHt)|psi(0)>
U = expm(-1j * t * H)
evolved_state = U @ up_up

# 8. Calculate probabilities of all four basis states
probabilities = np.abs(evolved_state)**2

# 9. Define an expectation-value function
def expectation(state, O):
    return np.vdot(state, O @ state)

# 10. Calculate the first spin's z-component
Sz_1 = Z0 / 2
sz_1 = expectation(evolved_state, Sz_1).real

# 11. Calculate the energy expectation value
energy = expectation(evolved_state, H).real

# 12. Print the results
print("Two-spin quantum sensor")
print("-----------------------")
print("Hilbert-space dimension:", len(evolved_state))
print("Initial state: |up up>")
print("Evolution time:", t)

print("\nMeasurement probabilities:")
labels = ["|up up>", "|up down>", "|down up>", "|down down>"]

for label, probability in zip(labels, probabilities):
    print(f"{label:12s}: {probability:.6f}")

print("\nExpectation value <Sz(1)>:", round(sz_1, 6))
print("Energy expectation <H>:", round(energy, 6))
print("Total probability:", probabilities.sum().real)
print("Normalization check:",
      "PASSED" if np.isclose(probabilities.sum(), 1.0) else "FAILED")



Two-spin quantum sensor
-----------------------
Hilbert-space dimension: 4
Initial state: |up up>
Evolution time: 1.0

Measurement probabilities:
|up up>     : 0.714033
|up down>   : 0.121960
|down up>   : 0.121960
|down down> : 0.042046

Expectation value <Sz(1)>: 0.335994
Energy expectation <H>: 1.0
Total probability: 1.0000000000000002
Normalization check: PASSED
